# Known Issues in TeAAL Specification

## Imports

Import the necessary modules.

In [1]:
# HiFiber boilerplate

from fibertree_bootstrap import *

fibertree_bootstrap(style="tree", animation='movie')

# Compilation boilerplate

import os
import sys
sys.path.insert(0, "../../")

from src import utils

Running bootstrap
The fibertree module is already installed and available to import


interactive(children=(Dropdown(description='style', options=('tree', 'uncompressed', 'tree+uncompressed'), val…

Button(description='Run all cells below', style=ButtonStyle())

## Initialization

Initialize the input tensors and variables needed for TeAAL specification.

For simplicity, the size of a thread warp is the same as the size of a thread block (`WARP_SIZE = BLOCK_SIZE`). Suppose that each GPU SM processes 1 thread warp/block per cycle.

In [2]:
M = 8
K = 8

# Tile Configuration
TILE_SIZE = 1 # Number of rows per tile
NUM_TILES = (M + TILE_SIZE - 1) // TILE_SIZE # Number of tiles

print(f"Tile Configuration\n \
        TILE_SIZE (Number of rows per tile): {TILE_SIZE} \n \
        NUM_TILES : {NUM_TILES}")

# GPU Kernel Configuration
BLOCK_SIZE = 2 # Number of threads per block
#GRID_SIZE = (M + BLOCK_SIZE - 1) // BLOCK_SIZE # Number of thread blocks
GRID_SIZE = 1
NUM_THREADS_PER_GRID = BLOCK_SIZE * GRID_SIZE
NUM_GRIDS = NUM_TILES // NUM_THREADS_PER_GRID

print(f"GPU Kernel Configuration\n \
        BLOCK_SIZE (Number of threads per block): {BLOCK_SIZE} \n \
        GRID_SIZE (Number of thread blocks): {GRID_SIZE}")
print(f"NUM_THREADS_PER_GRID: {NUM_THREADS_PER_GRID}")
print(f"NUM_GRIDS: {NUM_GRIDS}")

seed = 1

A_MK = Tensor.fromRandom(rank_ids=["M", "K"], shape=[M, K], seed=seed, density=[0.9, 0.6], name="A")
B_K = Tensor.fromRandom(rank_ids=["K"], shape=[K], seed=seed + 1, density=[1], name="B")

Tile Configuration
         TILE_SIZE (Number of rows per tile): 1 
         NUM_TILES : 8
GPU Kernel Configuration
         BLOCK_SIZE (Number of threads per block): 2 
         GRID_SIZE (Number of thread blocks): 1
NUM_THREADS_PER_GRID: 2
NUM_GRIDS: 4


## Issue 1: Incorrect Partitioning with uniform_shape()

Using uniform_shape() adds `depth=1` that leads to incorrect future partitioning.

In [ ]:
# WHAT WE GOT

Z_M2M1M0 = Tensor(rank_ids=["M2", "M1", "M0"], name="Z")
tmp0 = A_MK
tmp1 = tmp0.splitUniform(TILE_SIZE, depth=0)
tmp2 = tmp1.splitUniform(NUM_THREADS_PER_GRID, depth=1)

In [ ]:
# WHAT WE EXPECT

Z_M2M1M0 = Tensor(rank_ids=["M2", "M1", "M0"], name="Z")
tmp0 = A_MK
tmp1 = tmp0.splitUniform(TILE_SIZE, depth=0)
tmp2 = tmp1.splitUniform(NUM_THREADS_PER_GRID, depth=0)

## TeAAL Specifications

In [4]:
yaml = """
einsum:
  declaration:
    A: [M, K]
    B: [K]
    Z: [M]
  expressions:
    - Z[m] = A[m, k] * B[k]
mapping:
  rank-order:
    A: [M, K]
    B: [K]
    Z: [M]
  partitioning:
    Z:
      M: [uniform_shape(TILE_SIZE), uniform_shape(NUM_THREADS_PER_GRID)]
  loop-order:
    Z: [M2, M1, M0, K]
    # After First Partitioning:
    # M1: Number of tiles = NUM_TILES = 8
    # M0: Size of each tile = TILE_SIZE = 1

    # After Second Partitioning:
    # M2: Number of grids = 4
    # M1: Number of tiles per grid (Number of threads working) = 2
    # M0: Size of each tile = TILE_SIZE = 1
  spacetime:
    Z:
      space: [M1]
      time: [M2, M0, K]
"""

utils.compile(yaml)

## Issue 2: Unable to specify ranks on `slip: opt`

This prevents writing a TeAAL specification for GPU thread blocks being independent from each other. This means there exists a synchronization across the SMs.